In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# ============================================================
# CELL 1 — Reinstall (remplace l'ancienne CELL 1)
# ============================================================
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install transformers datasets scikit-learn --quiet

In [2]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import pandas as pd
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import os, warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {device}")
print(f"✅ GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

✅ Device : cuda
✅ GPU    : Tesla T4


In [3]:
# ============================================================
# CELL 3 — Chemins corrects
# ============================================================
PATH_DDINTER = "/kaggle/input/datasets/saadhachemkerfah/ddinte-r/ddinter_downloads_code_A.csv"
PATH_DB      = "/kaggle/input/datasets/saadhachemkerfah/data-pharm/db_drug_interactions.csv"
SAVE_PATH    = "/kaggle/working/medsafe_scibert"
MODEL_NAME   = "allenai/scibert_scivocab_uncased"

In [4]:
# ============================================================
# CELL 4 — Charger DDInter (labels officiels)
# ============================================================
ddinter = pd.read_csv(PATH_DDINTER)
print(f"DDInter shape : {ddinter.shape}")
print(f"Colonnes      : {ddinter.columns.tolist()}")
print(f"Labels uniques: {ddinter['Level'].unique()}")
print(f"\nDistribution :\n{ddinter['Level'].value_counts()}")

# Mapping officiel DDInter → 3 classes
LEVEL_MAP = {
    'Major':    2,   # DANGER
    'Moderate': 1,   # CAUTION
    'Minor':    0,   # MILD
    # Unknown → supprimé
}

ddinter = ddinter[ddinter['Level'] != 'Unknown'].reset_index(drop=True)
ddinter['label'] = ddinter['Level'].map(LEVEL_MAP)

# Texte : juste les noms (DDInter n'a pas de description)
ddinter['text'] = (
    ddinter['Drug_A'].str.lower().str.strip() +
    " interacts with " +
    ddinter['Drug_B'].str.lower().str.strip()
)

print(f"\n✅ DDInter après nettoyage : {ddinter.shape}")
print(ddinter['label'].value_counts().sort_index().rename({0:'MILD',1:'CAUTION',2:'DANGER'}))

DDInter shape : (56367, 5)
Colonnes      : ['DDInterID_A', 'Drug_A', 'DDInterID_B', 'Drug_B', 'Level']
Labels uniques: ['Moderate' 'Major' 'Minor' 'Unknown']

Distribution :
Level
Moderate    33057
Unknown     14767
Major        4906
Minor        3637
Name: count, dtype: int64

✅ DDInter après nettoyage : (41600, 7)
label
MILD        3637
CAUTION    33057
DANGER      4906
Name: count, dtype: int64


In [5]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if 'drug' in file.lower() or 'db_' in file.lower():
            print(os.path.join(root, file))

/kaggle/input/datasets/saadhachemkerfah/data-pharm/db_drug_interactions.csv


In [6]:
# ============================================================
# CELL 5 — Charger db_drug_interactions (descriptions riches)
# ============================================================
db = pd.read_csv(PATH_DB)
print(f"DB Drug shape : {db.shape}")
print(f"Colonnes      : {db.columns.tolist()}")

# Labellisation par patterns dans la description
def extract_severity(text: str) -> int:
    if not isinstance(text, str):
        return 1
    t = text.lower()

    danger_kw = [
        'cardiotoxic', 'nephrotoxic', 'hepatotoxic', 'neurotoxic',
        'myelosuppressive', 'bone marrow', 'qt interval',
        'serotonin syndrome', 'respiratory depression',
        'life-threatening', 'fatal', 'death', 'cardiac arrest',
        'anaphylaxis', 'hemorrhage', 'severe', 'toxic activities',
        'hypersensitivity reaction', 'torsades de pointes'
    ]
    mild_kw = [
        'photosensitizing activities', 'stimulating activities',
        'diagnostic activities', 'may increase the activities',
        'may decrease the activities', 'analgesic activities',
        'anti-inflammatory activities', 'sedative activities'
    ]

    if any(k in t for k in danger_kw):
        return 2   # DANGER
    elif any(k in t for k in mild_kw):
        return 0   # MILD
    else:
        return 1   # CAUTION (défaut)

db = db.dropna(subset=['Drug 1', 'Drug 2', 'Interaction Description']).reset_index(drop=True)
db['label'] = db['Interaction Description'].apply(extract_severity)

# Texte : noms + description (signal sémantique riche)
db['text'] = (
    db['Drug 1'].str.lower().str.strip() +
    " interacts with " +
    db['Drug 2'].str.lower().str.strip() +
    ": " +
    db['Interaction Description'].str.strip()
)

print(f"\n✅ DB Drug après nettoyage : {db.shape}")
print(db['label'].value_counts().sort_index().rename({0:'MILD',1:'CAUTION',2:'DANGER'}))

DB Drug shape : (191541, 3)
Colonnes      : ['Drug 1', 'Drug 2', 'Interaction Description']

✅ DB Drug après nettoyage : (191541, 5)
label
MILD         1870
CAUTION    187536
DANGER       2135
Name: count, dtype: int64


In [7]:
# ============================================================
# CELL 6b — Échantillon rapide 500 médicaments
# ============================================================

# Prendre les 500 médicaments les plus fréquents
top_drugs = pd.concat([
    ddinter['Drug_A'],
    ddinter['Drug_B']
]).value_counts().head(500).index.tolist()

# Filtrer DDInter pour garder seulement ces médicaments
df_fast = ddinter[
    ddinter['Drug_A'].isin(top_drugs) &
    ddinter['Drug_B'].isin(top_drugs)
][['text', 'label']].copy()

df_fast = df_fast.drop_duplicates(subset='text').reset_index(drop=True)

print(f"✅ Dataset rapide : {df_fast.shape}")
print(f"\n📊 Distribution :")
counts = df_fast['label'].value_counts().sort_index()
counts.index = ['MILD', 'CAUTION', 'DANGER']
print(counts)

df = df_fast  # ← remplace df principal

✅ Dataset rapide : (23341, 2)

📊 Distribution :
MILD        1429
CAUTION    19491
DANGER      2421
Name: count, dtype: int64


In [8]:
# ============================================================
# CELL 7 — Split
# ============================================================
X_train, X_val, y_train, y_val = train_test_split(
    df['text'].values,
    df['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label'].values
)

print(f"Train : {len(X_train):,} | Val : {len(X_val):,}")
print(f"Train dist : {np.bincount(y_train)}")
print(f"Val dist   : {np.bincount(y_val)}")

Train : 18,672 | Val : 4,669
Train dist : [ 1143 15592  1937]
Val dist   : [ 286 3899  484]


In [9]:
# ============================================================
# CELL 8 — Dataset + Tokenizer (max_len réduit = plus rapide)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DDIDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):  # ← 64 au lieu de 256
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(int(self.labels[idx]), dtype=torch.long)
        }

train_dataset = DDIDataset(X_train, y_train, tokenizer)
val_dataset   = DDIDataset(X_val,   y_val,   tokenizer)

print(f"✅ Train : {len(train_dataset):,} | Val : {len(val_dataset):,}")

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

✅ Train : 18,672 | Val : 4,669


In [10]:
# ============================================================
# CELL 9 — Modèle léger (DistilBERT au lieu de SciBERT)
# ============================================================
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""
device = torch.device("cpu")

# DistilBERT = 2x plus rapide que SciBERT, 40% moins de paramètres
FAST_MODEL = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    FAST_MODEL,
    num_labels=3,
    ignore_mismatched_sizes=True
)
model.to(device)
print(f"✅ Modèle : {FAST_MODEL}")
print(f"   Paramètres : {sum(p.numel() for p in model.parameters()):,}")

class_weights = compute_class_weight(
    'balanced',
    classes=np.array([0, 1, 2]),
    y=y_train
)
weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print(f"⚖️  Weights : MILD={class_weights[0]:.2f} | CAUTION={class_weights[1]:.2f} | DANGER={class_weights[2]:.2f}")

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = nn.CrossEntropyLoss(weight=weights_tensor)(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Modèle : distilbert-base-uncased
   Paramètres : 66,955,779
⚖️  Weights : MILD=5.45 | CAUTION=0.40 | DANGER=3.21


In [12]:
# ============================================================
# CELL 10 — TrainingArguments ultra rapide
# ============================================================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "macro_f1":   round(f1_score(labels, preds, average='macro'), 4),
        "danger_f1":  round(f1_score(labels, preds, average=None)[2], 4),
        "caution_f1": round(f1_score(labels, preds, average=None)[1], 4),
        "mild_f1":    round(f1_score(labels, preds, average=None)[0], 4),
    }

os.makedirs(SAVE_PATH, exist_ok=True)

training_args = TrainingArguments(
    output_dir            = "/kaggle/working/results",

    num_train_epochs              = 2,    # ← 2 epochs seulement
    per_device_train_batch_size   = 16,   # ← plus grand batch
    per_device_eval_batch_size    = 32,

    learning_rate         = 3e-5,         # ← lr plus élevé = converge vite
    weight_decay          = 0.01,
    warmup_ratio          = 0.1,
    optim                 = "adamw_torch",

    eval_strategy         = "epoch",
    save_strategy         = "epoch",
    load_best_model_at_end= True,
    metric_for_best_model = "macro_f1",
    greater_is_better     = True,
    save_total_limit      = 1,

    logging_steps         = 20,
    report_to             = "none",

    fp16                  = False,
    bf16                  = False,
    use_cpu               = True,
    dataloader_pin_memory = False,
    seed                  = 42,
)

trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=1)]
)

print("✅ Trainer prêt")
print(f"   Samples : {len(train_dataset):,}")
print(f"   Steps   : {len(train_dataset) // 16 * 2}")
print(f"   ⏱️  Temps estimé : ~10-15 min sur CPU")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


✅ Trainer prêt
   Samples : 18,672
   Steps   : 2334
   ⏱️  Temps estimé : ~10-15 min sur CPU


In [ ]:
# ============================================================
# CELL 11 — Entraînement
# ============================================================
print("🚀 Démarrage entraînement rapide...")
print(f"   Modèle  : {FAST_MODEL}")
print(f"   Train   : {len(train_dataset):,} paires")
print(f"   Val     : {len(val_dataset):,} paires")
print(f"   Epochs  : {training_args.num_train_epochs}")
print()

trainer.train()

print("\n✅ Entraînement terminé !")

🚀 Démarrage entraînement rapide...
   Modèle  : distilbert-base-uncased
   Train   : 18,672 paires
   Val     : 4,669 paires
   Epochs  : 2



Epoch,Training Loss,Validation Loss


In [16]:
# ============================================================
# CELL 12 — Évaluation finale
# ============================================================
predictions = trainer.predict(val_dataset)
y_pred = np.argmax(predictions.predictions, axis=1)

TARGET_NAMES = ['MILD', 'CAUTION', 'DANGER']

print("=" * 55)
print("  RAPPORT — DistilBERT MedSafe DDI")
print("=" * 55)
print(classification_report(y_val, y_pred, target_names=TARGET_NAMES))

macro_f1  = f1_score(y_val, y_pred, average='macro')
danger_f1 = f1_score(y_val, y_pred, average=None)[2]

print(f"🎯 Macro F1  : {macro_f1:.4f}")
print(f"🔴 DANGER F1 : {danger_f1:.4f}")

# Confusion matrix texte
cm = confusion_matrix(y_val, y_pred)
print(f"\nConfusion matrix :")
print(f"{'':10}", end="")
for n in TARGET_NAMES: print(f"{n:>10}", end="")
print()
for i, row in enumerate(cm):
    print(f"{TARGET_NAMES[i]:10}", end="")
    for v in row: print(f"{v:>10,}", end="")
    print()

  RAPPORT — DistilBERT MedSafe DDI
              precision    recall  f1-score   support

        MILD       0.53      0.79      0.63       286
     CAUTION       0.97      0.87      0.91      3899
      DANGER       0.55      0.86      0.67       484

    accuracy                           0.86      4669
   macro avg       0.68      0.84      0.74      4669
weighted avg       0.90      0.86      0.87      4669

🎯 Macro F1  : 0.7386
🔴 DANGER F1 : 0.6694

Confusion matrix :
                MILD   CAUTION    DANGER
MILD             227        45        14
CAUTION          199     3,375       325
DANGER             6        64       414


In [17]:
# ============================================================
# CELL 13 — Sauvegarde
# ============================================================
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✅ Modèle sauvegardé : {SAVE_PATH}")
print(f"   Fichiers : {os.listdir(SAVE_PATH)}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Modèle sauvegardé : /kaggle/working/medsafe_scibert
   Fichiers : ['tokenizer.json', 'tokenizer_config.json', 'model.safetensors', 'config.json', 'training_args.bin']


In [18]:
# ============================================================
# CELL 14 — Inférence FastAPI-ready
# ============================================================
from itertools import combinations

inf_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)
inf_model     = AutoModelForSequenceClassification.from_pretrained(SAVE_PATH)
inf_model.eval()
inf_model.to(torch.device("cpu"))

LABEL_MAP  = {0: 'MILD',    1: 'CAUTION', 2: 'DANGER'}
RISK_COLOR = {0: '🟢',      1: '🟡',      2: '🔴'}

def check_interactions(drug_list: list) -> list:
    results = []
    for d1, d2 in combinations(drug_list, 2):
        text = f"{d1.lower().strip()} interacts with {d2.lower().strip()}"
        enc  = inf_tokenizer(
            text,
            return_tensors = 'pt',
            max_length     = 64,
            truncation     = True,
            padding        = 'max_length'
        )

        with torch.no_grad():
            logits = inf_model(**enc).logits
            proba  = torch.softmax(logits, dim=1).cpu().numpy()[0]
            pred   = int(np.argmax(proba))

        results.append({
            'drug1':          d1,
            'drug2':          d2,
            'risk_level':     LABEL_MAP[pred],
            'icon':           RISK_COLOR[pred],
            'confidence':     round(float(proba[pred]), 3),
            'proba_mild':     round(float(proba[0]), 3),
            'proba_caution':  round(float(proba[1]), 3),
            'proba_danger':   round(float(proba[2]), 3),
            'send_to_gemini': pred > 0
        })

    results.sort(key=lambda x: -x['proba_danger'])
    return results


# ── Test ──────────────────────────────────────────────────
test_drugs = ['Aspirin', 'Warfarin', 'Ibuprofen', 'Metformin']
print(f"🧪 Test sur : {test_drugs}\n")

interactions = check_interactions(test_drugs)
for r in interactions:
    print(
        f"{r['icon']} {r['drug1']:15} + {r['drug2']:15} → "
        f"{r['risk_level']:8} (conf: {r['confidence']}) | "
        f"gemini: {r['send_to_gemini']}"
    )

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

🧪 Test sur : ['Aspirin', 'Warfarin', 'Ibuprofen', 'Metformin']

🟡 Warfarin        + Metformin       → CAUTION  (conf: 0.959) | gemini: True
🟡 Aspirin         + Warfarin        → CAUTION  (conf: 0.949) | gemini: True
🟡 Aspirin         + Metformin       → CAUTION  (conf: 0.962) | gemini: True
🟡 Warfarin        + Ibuprofen       → CAUTION  (conf: 0.939) | gemini: True
🟡 Ibuprofen       + Metformin       → CAUTION  (conf: 0.956) | gemini: True
🟡 Aspirin         + Ibuprofen       → CAUTION  (conf: 0.98) | gemini: True


In [19]:
# ============================================================
# CELL 15 — Exporter le modèle (téléchargement depuis Kaggle)
# ============================================================
import shutil

# Zipper le modèle pour téléchargement
zip_path = "/kaggle/working/medsafe_model"
shutil.make_archive(zip_path, 'zip', SAVE_PATH)

print(f"✅ Modèle zippé : {zip_path}.zip")
print(f"   Taille : {os.path.getsize(zip_path+'.zip') / 1024 / 1024:.1f} MB")
print(f"\n📥 Va dans Output → télécharge medsafe_model.zip")
print(f"   Ce fichier contient tout ce qu'il faut pour FastAPI")

✅ Modèle zippé : /kaggle/working/medsafe_model.zip
   Taille : 235.8 MB

📥 Va dans Output → télécharge medsafe_model.zip
   Ce fichier contient tout ce qu'il faut pour FastAPI


In [1]:
# ── Vérifier ce qui existe ──────────────────────────────────
import os

print("📂 /kaggle/working :")
for root, dirs, files in os.walk("/kaggle/working"):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path) / 1024 / 1024
        print(f"   {path}  ({size:.1f} MB)")

📂 /kaggle/working :
   /kaggle/working/.virtual_documents/__notebook_source__.ipynb  (0.0 MB)


In [2]:
# ── Relancer la sauvegarde manuellement ────────────────────
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ Sauvegardé dans : {SAVE_PATH}")
print(f"   Fichiers : {os.listdir(SAVE_PATH)}")

NameError: name 'trainer' is not defined